In [41]:
#1. 데이터셋 다운로드
import os
import urllib.request
import zipfile

os.makedirs('/content', exist_ok=True)

urllib.request.urlretrieve("http://files.grouplens.org/datasets/movielens/ml-100k.zip", "/content/ml-100k.zip")

with zipfile.ZipFile("/content/ml-100k.zip", 'r') as zip_ref:
    zip_ref.extractall("/content/")

In [42]:
# 2. 라이브러리 임포트
import math
import random
import pickle
from collections import defaultdict
from pathlib import Path
import numpy as np

In [43]:
# 3. 데이터 로드 함수
def read_ratings(path):
    rows = []
    with open(path, 'r', encoding='latin-1') as f:
        for line in f:
            user_id, item_id, rating, timestamp = line.strip().split('\t')
            rows.append((int(user_id), int(item_id), float(rating), int(timestamp)))
    return rows

def read_items(path):
    titles = {}
    genres = {}
    with open(path, 'r', encoding='latin-1') as f:
        for line in f:
            parts = line.rstrip('\n').split('|')
            iid = int(parts[0])
            titles[iid] = parts[1]
            genre_vec = np.array([int(x) for x in parts[5:24]], dtype=np.float32)
            norm = np.linalg.norm(genre_vec)
            genres[iid] = genre_vec / norm if norm > 0 else genre_vec
    return titles, genres

def split_validation_from_base(base_rows, valid_ratio=0.1, seed=42):
    by_user = defaultdict(list)
    for row in base_rows:
        by_user[row[0]].append(row)
    inner_train_rows, valid_rows = [], []
    for user_rows in by_user.values():
        sorted_rows = sorted(user_rows, key=lambda x: x[3])
        n_valid = max(1, int(len(sorted_rows) * valid_ratio)) if len(sorted_rows) >= 5 else 0
        valid_rows.extend(sorted_rows[-n_valid:] if n_valid > 0 else [])
        inner_train_rows.extend(sorted_rows[:-n_valid] if n_valid > 0 else sorted_rows)
    return inner_train_rows, valid_rows

In [44]:
# 4. ItemKNNRecommender 클래스
class ItemKNNRecommender:
    def __init__(self, k=40, shrinkage=25.0, min_candidate_popularity=5):
        self.k = k
        self.shrinkage = shrinkage
        self.min_candidate_popularity = min_candidate_popularity

    def fit(self, rows):
        self.users = sorted({u for u, _, _, _ in rows})
        self.items = sorted({i for _, i, _, _ in rows})
        self.user_to_idx = {u: idx for idx, u in enumerate(self.users)}
        self.item_to_idx = {i: idx for idx, i in enumerate(self.items)}
        n_users, n_items = len(self.users), len(self.items)
        self.ratings = np.full((n_users, n_items), np.nan, dtype=np.float32)
        self.user_rated_items = defaultdict(list)
        self.item_popularity = defaultdict(int)
        for user_id, item_id, rating, _ in rows:
            uidx = self.user_to_idx[user_id]
            iidx = self.item_to_idx[item_id]
            self.ratings[uidx, iidx] = rating
            self.user_rated_items[user_id].append((item_id, rating))
            self.item_popularity[item_id] += 1
        self.global_mean = float(np.nanmean(self.ratings))
        self.user_means = np.nanmean(self.ratings, axis=1)
        self.item_means = np.nanmean(self.ratings, axis=0)
        self.user_means = np.where(np.isnan(self.user_means), self.global_mean, self.user_means)
        self.item_means = np.where(np.isnan(self.item_means), self.global_mean, self.item_means)
        mask = ~np.isnan(self.ratings)
        centered = np.where(mask, self.ratings - self.user_means[:, None], 0.0)
        numerator = centered.T @ centered
        norms = np.sqrt(np.sum(centered * centered, axis=0))
        denominator = norms[:, None] * norms[None, :]
        similarity = np.divide(numerator, denominator,
            out=np.zeros_like(numerator, dtype=np.float32), where=denominator > 0)
        co_counts = mask.astype(np.float32).T @ mask.astype(np.float32)
        significance = co_counts / (co_counts + self.shrinkage)
        self.similarity = similarity * significance
        np.fill_diagonal(self.similarity, 0.0)
        self.neighbors = {}
        for item_idx in range(n_items):
            sims = self.similarity[item_idx]
            order = np.argsort(np.abs(sims))[::-1]
            order = [idx for idx in order if sims[idx] != 0.0][: self.k]
            self.neighbors[item_idx] = order
        return self

    def predict(self, user_id, item_id):
        if user_id not in self.user_to_idx:
            return self._clip(self.item_mean(item_id))
        if item_id not in self.item_to_idx:
            return self._clip(self.user_mean(user_id))
        uidx = self.user_to_idx[user_id]
        target_idx = self.item_to_idx[item_id]
        baseline = self.user_means[uidx]
        numerator = denominator = 0.0
        for neighbor_idx in self.neighbors[target_idx]:
            rating = self.ratings[uidx, neighbor_idx]
            if np.isnan(rating):
                continue
            sim = float(self.similarity[target_idx, neighbor_idx])
            numerator += sim * (float(rating) - baseline)
            denominator += abs(sim)
        if denominator == 0.0:
            return self._clip(0.6 * baseline + 0.4 * self.item_means[target_idx])
        return self._clip(baseline + numerator / denominator)

    def recommend(self, user_id, top_n=10):
        if user_id not in self.user_to_idx:
            return self.popular_items(top_n)
        seen = {item_id for item_id, _ in self.user_rated_items[user_id]}
        candidates = [
            item_id for item_id in self.items
            if item_id not in seen
            and self.item_popularity.get(item_id, 0) >= self.min_candidate_popularity
        ]
        scored = [(item_id, self.predict(user_id, item_id)) for item_id in candidates]
        scored.sort(key=lambda x: (x[1], self.item_popularity.get(x[0], 0)), reverse=True)
        return scored[:top_n]

    def popular_items(self, top_n):
        return [(item_id, self.item_mean(item_id))
            for item_id, _ in sorted(self.item_popularity.items(),
                key=lambda x: (x[1], self.item_mean(x[0])), reverse=True)[:top_n]]

    def user_mean(self, user_id):
        if user_id not in self.user_to_idx:
            return self.global_mean
        return float(self.user_means[self.user_to_idx[user_id]])

    def item_mean(self, item_id):
        if item_id not in self.item_to_idx:
            return self.global_mean
        return float(self.item_means[self.item_to_idx[item_id]])

    @staticmethod
    def _clip(value): return min(5.0, max(1.0, float(value)))

In [45]:
# 5. MFRecommender 클래스
class MFRecommender:
    def __init__(self, n_factors=20, lr=0.005, reg=0.02, n_epochs=30,
                 min_candidate_popularity=1, seed=42):
        self.n_factors = n_factors
        self.lr = lr
        self.reg = reg
        self.n_epochs = n_epochs
        self.min_candidate_popularity = min_candidate_popularity
        self.seed = seed

    def fit(self, train_rows):
        self.users = sorted({u for u, _, _, _ in train_rows})
        self.items = sorted({i for _, i, _, _ in train_rows})
        self.user_to_idx = {u: idx for idx, u in enumerate(self.users)}
        self.item_to_idx = {i: idx for idx, i in enumerate(self.items)}
        n_users, n_items = len(self.users), len(self.items)
        self.user_rated_items = defaultdict(list)
        self.item_popularity  = defaultdict(int)
        rating_sum = 0.0
        for user_id, item_id, rating, _ in train_rows:
            self.user_rated_items[user_id].append((item_id, rating))
            self.item_popularity[item_id] += 1
            rating_sum += rating
        self.global_mean = rating_sum / len(train_rows) if train_rows else 3.0
        rng = np.random.default_rng(self.seed)
        self.U   = rng.normal(0, 0.01, (n_users, self.n_factors)).astype(np.float32)
        self.V   = rng.normal(0, 0.01, (n_items, self.n_factors)).astype(np.float32)
        self.b_u = np.zeros(n_users, dtype=np.float32)
        self.b_i = np.zeros(n_items, dtype=np.float32)
        train_list = [(self.user_to_idx[u], self.item_to_idx[i], r) for u, i, r, _ in train_rows]
        rng_shuffle = random.Random(self.seed)
        for epoch in range(self.n_epochs):
            rng_shuffle.shuffle(train_list)
            sq_err = 0.0
            for uidx, iidx, rating in train_list:
                pred = (self.global_mean + self.b_u[uidx] + self.b_i[iidx]
                        + float(self.U[uidx] @ self.V[iidx]))
                err    = rating - pred
                sq_err += err * err
                v_old = self.V[iidx].copy()
                self.V[iidx]   += self.lr * (err * self.U[uidx] - self.reg * self.V[iidx])
                self.U[uidx]   += self.lr * (err * v_old        - self.reg * self.U[uidx])
                self.b_u[uidx] += self.lr * (err - self.reg * self.b_u[uidx])
                self.b_i[iidx] += self.lr * (err - self.reg * self.b_i[iidx])
            if (epoch + 1) % 5 == 0:
                print(f'  epoch {epoch+1:>2}/{self.n_epochs}  train RMSE={math.sqrt(sq_err/len(train_list)):.4f}')
        return self

    def predict(self, user_id, item_id):
        has_u = user_id in self.user_to_idx
        has_i = item_id in self.item_to_idx
        if not has_u and not has_i: return self._clip(self.global_mean)
        if not has_u: return self._clip(self.global_mean + self.b_i[self.item_to_idx[item_id]])
        if not has_i: return self._clip(self.global_mean + self.b_u[self.user_to_idx[user_id]])
        uidx, iidx = self.user_to_idx[user_id], self.item_to_idx[item_id]
        return self._clip(self.global_mean + self.b_u[uidx] + self.b_i[iidx]
                          + float(self.U[uidx] @ self.V[iidx]))

    def recommend(self, user_id, top_n=10):
        if user_id not in self.user_to_idx:
            return self.popular_items(top_n)
        seen = {iid for iid, _ in self.user_rated_items[user_id]}
        candidates = [iid for iid in self.items
                      if iid not in seen
                      and self.item_popularity.get(iid, 0) >= self.min_candidate_popularity]
        scored = [(iid, self.predict(user_id, iid)) for iid in candidates]
        scored.sort(key=lambda x: (x[1], self.item_popularity.get(x[0], 0)), reverse=True)
        return scored[:top_n]

    def popular_items(self, top_n=10):
        sorted_items = sorted(self.item_popularity.keys(),
            key=lambda iid: (self.item_popularity[iid], self.item_mean(iid)), reverse=True)
        return [(iid, self.item_mean(iid)) for iid in sorted_items[:top_n]]

    def item_mean(self, item_id):
        if item_id not in self.item_to_idx: return self.global_mean
        return self._clip(self.global_mean + self.b_i[self.item_to_idx[item_id]])

    @staticmethod
    def _clip(value): return min(5.0, max(1.0, float(value)))

In [46]:
# 6. EnsembleRecommender 클래스
class EnsembleRecommender:
    def __init__(self, knn_model, mf_model, weight_knn=0.5, min_candidate_popularity=1):
        self.knn = knn_model
        self.mf  = mf_model
        self.w_knn = weight_knn
        self.w_mf  = 1.0 - weight_knn
        self.min_candidate_popularity = min_candidate_popularity
        self.items = sorted(set(knn_model.items) | set(mf_model.items))
        self.item_popularity = mf_model.item_popularity
        self.user_rated_items = defaultdict(set)
        for uid, rated in knn_model.user_rated_items.items():
            for item_id, _ in rated:
                self.user_rated_items[uid].add(item_id)
        for uid, rated in mf_model.user_rated_items.items():
            for item_id, _ in rated:
                self.user_rated_items[uid].add(item_id)
        self.users = sorted(set(knn_model.users) | set(mf_model.users))
        self.user_to_idx = {u: idx for idx, u in enumerate(self.users)}

    def predict(self, user_id, item_id):
        return min(5.0, max(1.0,
            self.w_knn * self.knn.predict(user_id, item_id) +
            self.w_mf  * self.mf.predict(user_id, item_id)))

    def recommend(self, user_id, top_n=10):
        seen = self.user_rated_items.get(user_id, set())
        candidates = [
            iid for iid in self.items
            if iid not in seen
            and self.item_popularity.get(iid, 0) >= self.min_candidate_popularity
        ]
        scored = [(iid, self.predict(user_id, iid)) for iid in candidates]
        scored.sort(key=lambda x: (x[1], self.item_popularity.get(x[0], 0)), reverse=True)
        return scored[:top_n]

In [47]:
# 7. 평가 함수
def rating_metrics(model, rows):
    sq_errors, abs_errors = [], []
    for user_id, item_id, rating, _ in rows:
        pred = model.predict(user_id, item_id)
        e = rating - pred
        sq_errors.append(e * e)
        abs_errors.append(abs(e))
    return {
        'rmse': math.sqrt(sum(sq_errors) / len(sq_errors)),
        'mae':  sum(abs_errors) / len(abs_errors),
    }

def tune_knn(inner_train_rows, valid_rows, k_values, shrinkage, min_pop):
    results = []
    for k in k_values:
        print(f'  K={k}')
        model = ItemKNNRecommender(k=k, shrinkage=shrinkage,
                                   min_candidate_popularity=min_pop).fit(inner_train_rows)
        m = rating_metrics(model, valid_rows)
        results.append((k, m['rmse'], m['mae']))
        print(f'    RMSE={m["rmse"]:.4f}  MAE={m["mae"]:.4f}')
    return min(results, key=lambda x: x[1]), results

def tune_factors(inner_train_rows, valid_rows, factor_values, lr, reg, n_epochs, min_pop):
    results = []
    for n_factors in factor_values:
        print(f'  n_factors={n_factors}')
        model = MFRecommender(n_factors=n_factors, lr=lr, reg=reg,
                              n_epochs=n_epochs, min_candidate_popularity=min_pop).fit(inner_train_rows)
        m = rating_metrics(model, valid_rows)
        results.append((n_factors, m['rmse'], m['mae']))
        print(f'    RMSE={m["rmse"]:.4f}  MAE={m["mae"]:.4f}')
    return min(results, key=lambda x: x[1]), results

def tune_ensemble_weight(knn_model, mf_model, valid_rows, weight_steps=11, min_pop=1):
    results = []
    for i in range(weight_steps):
        w_knn = round(i / (weight_steps - 1), 1)
        ens = EnsembleRecommender(knn_model, mf_model, weight_knn=w_knn, min_candidate_popularity=min_pop)
        m = rating_metrics(ens, valid_rows)
        results.append((w_knn, m['rmse'], m['mae']))
        print(f'  w_knn={w_knn:.1f}  w_mf={1-w_knn:.1f}  RMSE={m["rmse"]:.4f}  MAE={m["mae"]:.4f}')
    return min(results, key=lambda x: x[1]), results

In [48]:
# 8. 설정
K_VALUES                 = [20, 40, 80]
FACTOR_VALUES            = [10, 20, 50, 100]
SHRINKAGE                = 25.0
LR                       = 0.005
REG                      = 0.02
N_EPOCHS                 = 30
MIN_CANDIDATE_POPULARITY = 1
DATA_DIR                 = Path('ml-100k')

In [49]:
# 9. 데이터 로드 및 분할
base_rows      = read_ratings(DATA_DIR / 'ua.base')
titles, genres = read_items(DATA_DIR / 'u.item')

inner_train_rows, valid_rows = split_validation_from_base(base_rows)
print(f'ua.base: {len(base_rows):,}  |  inner_train: {len(inner_train_rows):,}  |  valid: {len(valid_rows):,}')

ua.base: 90,570  |  inner_train: 81,917  |  valid: 8,653


In [50]:
# 10. KNN K 튜닝
print('[튜닝] K 탐색')
best_knn, knn_valid_results = tune_knn(
    inner_train_rows, valid_rows, K_VALUES, SHRINKAGE, MIN_CANDIDATE_POPULARITY
)
best_k = best_knn[0]
print(f'\n→ 선택: K={best_k}')

[튜닝] K 탐색
  K=20
    RMSE=1.0700  MAE=0.8263
  K=40
    RMSE=1.0430  MAE=0.8120
  K=80
    RMSE=1.0341  MAE=0.8100

→ 선택: K=80


In [51]:
# 11. MF n_factors 튜닝
print('[튜닝] n_factors 탐색')
best_mf, mf_valid_results = tune_factors(
    inner_train_rows, valid_rows, FACTOR_VALUES, LR, REG, N_EPOCHS, MIN_CANDIDATE_POPULARITY
)
best_factors = best_mf[0]
print(f'\n→ 선택: n_factors={best_factors}')

[튜닝] n_factors 탐색
  n_factors=10
  epoch  5/30  train RMSE=0.9373
  epoch 10/30  train RMSE=0.9229
  epoch 15/30  train RMSE=0.9175
  epoch 20/30  train RMSE=0.9143
  epoch 25/30  train RMSE=0.9102
  epoch 30/30  train RMSE=0.8998
    RMSE=0.9794  MAE=0.7804
  n_factors=20
  epoch  5/30  train RMSE=0.9372
  epoch 10/30  train RMSE=0.9228
  epoch 15/30  train RMSE=0.9173
  epoch 20/30  train RMSE=0.9138
  epoch 25/30  train RMSE=0.9088
  epoch 30/30  train RMSE=0.8963
    RMSE=0.9796  MAE=0.7806
  n_factors=50
  epoch  5/30  train RMSE=0.9371
  epoch 10/30  train RMSE=0.9225
  epoch 15/30  train RMSE=0.9164
  epoch 20/30  train RMSE=0.9111
  epoch 25/30  train RMSE=0.8999
  epoch 30/30  train RMSE=0.8771
    RMSE=0.9723  MAE=0.7738
  n_factors=100
  epoch  5/30  train RMSE=0.9369
  epoch 10/30  train RMSE=0.9219
  epoch 15/30  train RMSE=0.9151
  epoch 20/30  train RMSE=0.9073
  epoch 25/30  train RMSE=0.8898
  epoch 30/30  train RMSE=0.8595
    RMSE=0.9692  MAE=0.7712

→ 선택: n_factors=

In [52]:
# 12. 앙상블 가중치 튜닝
print('[튜닝] 앙상블 가중치 탐색')
tmp_knn = ItemKNNRecommender(k=best_k, shrinkage=SHRINKAGE,
                              min_candidate_popularity=MIN_CANDIDATE_POPULARITY).fit(inner_train_rows)
tmp_mf  = MFRecommender(n_factors=best_factors, lr=LR, reg=REG,
                         n_epochs=N_EPOCHS, min_candidate_popularity=MIN_CANDIDATE_POPULARITY).fit(inner_train_rows)
best_w, weight_results = tune_ensemble_weight(tmp_knn, tmp_mf, valid_rows, min_pop=MIN_CANDIDATE_POPULARITY)
best_w_knn = best_w[0]
print(f'\n→ 선택: w_knn={best_w_knn:.1f}  w_mf={1-best_w_knn:.1f}')

del valid_rows, tmp_knn, tmp_mf

[튜닝] 앙상블 가중치 탐색
  epoch  5/30  train RMSE=0.9369
  epoch 10/30  train RMSE=0.9219
  epoch 15/30  train RMSE=0.9151
  epoch 20/30  train RMSE=0.9073
  epoch 25/30  train RMSE=0.8898
  epoch 30/30  train RMSE=0.8595
  w_knn=0.0  w_mf=1.0  RMSE=0.9692  MAE=0.7712
  w_knn=0.1  w_mf=0.9  RMSE=0.9681  MAE=0.7697
  w_knn=0.2  w_mf=0.8  RMSE=0.9688  MAE=0.7694
  w_knn=0.3  w_mf=0.7  RMSE=0.9712  MAE=0.7704
  w_knn=0.4  w_mf=0.6  RMSE=0.9753  MAE=0.7728
  w_knn=0.5  w_mf=0.5  RMSE=0.9811  MAE=0.7763
  w_knn=0.6  w_mf=0.4  RMSE=0.9885  MAE=0.7809
  w_knn=0.7  w_mf=0.3  RMSE=0.9976  MAE=0.7866
  w_knn=0.8  w_mf=0.2  RMSE=1.0083  MAE=0.7935
  w_knn=0.9  w_mf=0.1  RMSE=1.0204  MAE=0.8013
  w_knn=1.0  w_mf=0.0  RMSE=1.0341  MAE=0.8100

→ 선택: w_knn=0.1  w_mf=0.9


In [53]:
# 13. 최종 모델 학습
print(f'[최종 학습] KNN  K={best_k}')
final_knn = ItemKNNRecommender(k=best_k, shrinkage=SHRINKAGE,
                                min_candidate_popularity=MIN_CANDIDATE_POPULARITY).fit(base_rows)

print(f'[최종 학습] MF  n_factors={best_factors}')
final_mf = MFRecommender(n_factors=best_factors, lr=LR, reg=REG,
                          n_epochs=N_EPOCHS, min_candidate_popularity=MIN_CANDIDATE_POPULARITY).fit(base_rows)

final_ensemble = EnsembleRecommender(final_knn, final_mf, weight_knn=best_w_knn,
                                      min_candidate_popularity=MIN_CANDIDATE_POPULARITY)
print('앙상블 생성 완료')

[최종 학습] KNN  K=80
[최종 학습] MF  n_factors=100
  epoch  5/30  train RMSE=0.9395
  epoch 10/30  train RMSE=0.9250
  epoch 15/30  train RMSE=0.9185
  epoch 20/30  train RMSE=0.9093
  epoch 25/30  train RMSE=0.8879
  epoch 30/30  train RMSE=0.8556
앙상블 생성 완료


In [54]:
# 14. 모델 저장
import os
save_dir = 'saved_models'
os.makedirs(save_dir, exist_ok=True)

bundle = {
    'knn':               final_knn,
    'mf':                final_mf,
    'ensemble':          final_ensemble,
    'titles':            titles,
    'genres':            genres,
    'best_k':            best_k,
    'best_factors':      best_factors,
    'best_w_knn':        best_w_knn,
    'knn_valid_results': knn_valid_results,
    'mf_valid_results':  mf_valid_results,
    'weight_results':    weight_results,
}
save_path = f'{save_dir}/ensemble_model.pkl'
with open(save_path, 'wb') as f:
    pickle.dump(bundle, f)
print(f'저장 완료: {save_path}')

저장 완료: saved_models/ensemble_model.pkl
